# Notebook 06: Build a RAG Pipeline from Scratch

> **Goal:** Understand every component of a RAG (Retrieval-Augmented Generation) system by building one from scratch — without hiding anything behind LangChain abstractions.

**What you'll build:**
1. A document chunker
2. An embedding-based vector store
3. A hybrid retriever (BM25 + semantic)
4. A RAG Q&A pipeline
5. Evaluation with RAGAS metrics

**Time:** ~4 hours

**Context:** You work for a company and need to answer questions from the company's internal documents.

In [ ]:
# Install required packages
# !pip install openai numpy tiktoken rank-bm25 ragas datasets langchain langchain-openai

import os
import json
import numpy as np
from openai import OpenAI
from dataclasses import dataclass
from typing import List, Tuple, Optional
import re

client = OpenAI(api_key=os.environ.get('OPENAI_API_KEY'))
print('Setup complete!')

## Part 1: Documents and Chunking

Why chunk? LLMs have context window limits. Also, smaller chunks = more precise retrieval.

In [ ]:
# Sample documents (in real life, load from PDFs, web, databases)
DOCUMENTS = [
    {
        'title': 'Python History',
        'content': '''Python is a high-level, general-purpose programming language.
Its design philosophy emphasizes code readability with the use of significant indentation.

Python was created by Guido van Rossum and first released in 1991. Van Rossum remained
the principal author of Python until he stepped down in July 2018. He was benevolent
dictator for life (BDFL) of the Python community for many years.

Python 2.0 was released in 2000 and introduced new features such as list comprehensions,
garbage collection, and Unicode support. Python 3.0 was released in 2008 and was a
major revision not backward compatible with Python 2.x.

Python is dynamically typed and garbage-collected. It supports multiple programming
paradigms, including structured, object-oriented, and functional programming.'''
    },
    {
        'title': 'Machine Learning Overview',
        'content': '''Machine learning (ML) is a field of study in artificial intelligence
concerned with the development and study of statistical algorithms that can learn from
data and generalize to unseen data, and thus perform tasks without explicit instructions.

The three main categories of machine learning are:
1. Supervised learning: The algorithm learns from labeled training data. Examples include
   classification (predicting categories) and regression (predicting continuous values).
2. Unsupervised learning: The algorithm finds patterns in unlabeled data. Examples
   include clustering and dimensionality reduction.
3. Reinforcement learning: An agent learns by interacting with an environment, receiving
   rewards or penalties for its actions.

Key ML algorithms include: Linear Regression, Logistic Regression, Decision Trees,
Random Forests, Gradient Boosting, Support Vector Machines, and Neural Networks.'''
    },
    {
        'title': 'Large Language Models',
        'content': '''Large language models (LLMs) are neural network models with billions
of parameters, trained on massive amounts of text data. They can generate human-like text,
answer questions, write code, and perform many other language tasks.

The transformer architecture, introduced in the paper "Attention is All You Need" (2017),
is the foundation of modern LLMs. Key components include:
- Self-attention mechanism: allows the model to weigh relationships between words
- Positional encoding: provides sequence order information
- Feed-forward networks: transform token representations layer by layer

Notable LLMs include GPT-4 (OpenAI), Claude (Anthropic), Gemini (Google), and Llama
(Meta). GPT-4 was released in March 2023. Claude was developed with a focus on safety
and helpful, harmless, honest behavior.

RAG (Retrieval-Augmented Generation) combines LLMs with external knowledge retrieval
to reduce hallucinations and enable access to current information.'''
    },
]

print(f'Loaded {len(DOCUMENTS)} documents')
for doc in DOCUMENTS:
    words = len(doc['content'].split())
    print(f"  '{doc['title']}': {words} words")

In [ ]:
@dataclass
class Chunk:
    """A piece of a document with metadata."""
    text: str
    doc_title: str
    chunk_id: int
    char_start: int
    char_end: int


def recursive_text_splitter(
    text: str,
    doc_title: str,
    chunk_size: int = 300,
    chunk_overlap: int = 50,
    separators: List[str] = ['\n\n', '\n', '. ', ' ', ''],
) -> List[Chunk]:
    """
    Split text recursively using decreasing separator sizes.
    First tries to split on paragraphs (\n\n), then sentences, then words.
    This preserves semantic boundaries better than fixed-size splitting.
    """
    chunks = []

    def _split(text: str, start_pos: int = 0):
        if len(text) <= chunk_size:
            if text.strip():
                chunks.append(Chunk(
                    text=text.strip(),
                    doc_title=doc_title,
                    chunk_id=len(chunks),
                    char_start=start_pos,
                    char_end=start_pos + len(text),
                ))
            return

        # Find the best separator
        for sep in separators:
            if sep in text:
                # Split at the separator closest to chunk_size
                parts = text.split(sep)
                current = ''
                current_start = start_pos

                for part in parts:
                    candidate = current + sep + part if current else part
                    if len(candidate) > chunk_size and current:
                        _split(current, current_start)
                        # Overlap: include end of current chunk at start of next
                        overlap_text = current[-chunk_overlap:] if len(current) > chunk_overlap else current
                        current = overlap_text + sep + part
                        current_start = start_pos + len(current) - len(part)
                    else:
                        current = candidate

                if current:
                    _split(current, current_start)
                return

        # No separator found — hard split
        chunks.append(Chunk(
            text=text[:chunk_size].strip(),
            doc_title=doc_title,
            chunk_id=len(chunks),
            char_start=start_pos,
            char_end=start_pos + chunk_size,
        ))

    _split(text)
    return chunks


# Chunk all documents
all_chunks = []
for doc in DOCUMENTS:
    chunks = recursive_text_splitter(
        doc['content'], doc['title'],
        chunk_size=300, chunk_overlap=50
    )
    all_chunks.extend(chunks)
    print(f"'{doc['title']}': {len(chunks)} chunks")

print(f'\nTotal chunks: {len(all_chunks)}')
print('\nExample chunk:')
print('-' * 60)
print(all_chunks[0].text)

## Part 2: Embedding-Based Vector Store

In [ ]:
class SimpleVectorStore:
    """Minimal vector store — understand what LangChain's Chroma/FAISS does."""

    EMBEDDING_MODEL = 'text-embedding-3-small'  # 1536 dimensions, cheap

    def __init__(self):
        self.chunks: List[Chunk] = []
        self.embeddings: List[np.ndarray] = []

    def embed_text(self, text: str) -> np.ndarray:
        """Get embedding vector for a text string."""
        response = client.embeddings.create(
            model=self.EMBEDDING_MODEL,
            input=text,
        )
        return np.array(response.data[0].embedding)

    def embed_batch(self, texts: List[str]) -> List[np.ndarray]:
        """Embed multiple texts efficiently in one API call."""
        response = client.embeddings.create(
            model=self.EMBEDDING_MODEL,
            input=texts,
        )
        return [np.array(d.embedding) for d in response.data]

    def add_chunks(self, chunks: List[Chunk], batch_size: int = 100):
        """Embed and store document chunks."""
        print(f'Embedding {len(chunks)} chunks...')

        for i in range(0, len(chunks), batch_size):
            batch = chunks[i:i + batch_size]
            texts = [c.text for c in batch]
            embeddings = self.embed_batch(texts)

            self.chunks.extend(batch)
            self.embeddings.extend(embeddings)
            print(f'  Embedded {min(i+batch_size, len(chunks))}/{len(chunks)} chunks')

    @staticmethod
    def cosine_similarity(a: np.ndarray, b: np.ndarray) -> float:
        """Dot product of unit vectors = cosine similarity."""
        return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-8))

    def similarity_search(
        self, query: str, k: int = 5
    ) -> List[Tuple[Chunk, float]]:
        """Find k most similar chunks to the query."""
        query_emb = self.embed_text(query)

        scores = [
            (chunk, self.cosine_similarity(query_emb, emb))
            for chunk, emb in zip(self.chunks, self.embeddings)
        ]

        return sorted(scores, key=lambda x: x[1], reverse=True)[:k]


# Build the index
vector_store = SimpleVectorStore()
vector_store.add_chunks(all_chunks)

print(f'\nVector store ready: {len(vector_store.chunks)} vectors')
print(f'Embedding dimensions: {len(vector_store.embeddings[0])}')

In [ ]:
# Test retrieval
query = 'Who created Python and when was it released?'
results = vector_store.similarity_search(query, k=3)

print(f'Query: "{query}"')
print('=' * 60)
for i, (chunk, score) in enumerate(results, 1):
    print(f'\nResult {i} (score: {score:.4f}) — from "{chunk.doc_title}"')
    print(chunk.text[:200] + '...')

## Part 3: BM25 + Hybrid Retrieval

Semantic search is great for meaning. BM25 is great for exact terms. Hybrid combines both.

In [ ]:
from rank_bm25 import BM25Okapi

class HybridRetriever:
    """
    Combines BM25 (keyword) + semantic (embedding) search.
    Uses Reciprocal Rank Fusion (RRF) to merge result lists.
    """

    def __init__(self, vector_store: SimpleVectorStore, bm25_weight: float = 0.4):
        self.vector_store = vector_store
        self.bm25_weight = bm25_weight  # 40% BM25, 60% semantic
        self.semantic_weight = 1 - bm25_weight

        # Build BM25 index
        tokenized_corpus = [
            chunk.text.lower().split()
            for chunk in vector_store.chunks
        ]
        self.bm25 = BM25Okapi(tokenized_corpus)
        print(f'BM25 index built: {len(vector_store.chunks)} documents')

    def reciprocal_rank_fusion(
        self,
        ranked_lists: List[List[Tuple[int, float]]],
        k: int = 60,
    ) -> List[Tuple[int, float]]:
        """
        RRF combines multiple ranked lists.
        Score = sum(1 / (k + rank)) across all lists.
        k=60 is the standard constant (Cormack et al., 2009).
        """
        scores = {}
        weights = [self.bm25_weight, self.semantic_weight]

        for ranked_list, weight in zip(ranked_lists, weights):
            for rank, (doc_idx, _) in enumerate(ranked_list):
                rrf_score = weight / (k + rank + 1)
                scores[doc_idx] = scores.get(doc_idx, 0) + rrf_score

        return sorted(scores.items(), key=lambda x: x[1], reverse=True)

    def retrieve(self, query: str, k: int = 5) -> List[Tuple[Chunk, float]]:
        """Hybrid search: BM25 + semantic, merged with RRF."""
        chunks = self.vector_store.chunks

        # BM25 retrieval
        bm25_scores = self.bm25.get_scores(query.lower().split())
        bm25_ranked = sorted(
            enumerate(bm25_scores), key=lambda x: x[1], reverse=True
        )[:k * 3]

        # Semantic retrieval
        query_emb = self.vector_store.embed_text(query)
        semantic_ranked = sorted(
            enumerate([
                self.vector_store.cosine_similarity(query_emb, emb)
                for emb in self.vector_store.embeddings
            ]),
            key=lambda x: x[1],
            reverse=True
        )[:k * 3]

        # Merge with RRF
        fused = self.reciprocal_rank_fusion([bm25_ranked, semantic_ranked])

        return [(chunks[idx], score) for idx, score in fused[:k]]


retriever = HybridRetriever(vector_store, bm25_weight=0.4)

# Compare semantic vs hybrid
query = 'transformer attention mechanism'
print(f'Query: "{query}"')

print('\n--- SEMANTIC ONLY ---')
for chunk, score in vector_store.similarity_search(query, k=2):
    print(f'[{score:.4f}] {chunk.text[:100]}...')

print('\n--- HYBRID (BM25 + Semantic) ---')
for chunk, score in retriever.retrieve(query, k=2):
    print(f'[{score:.4f}] {chunk.text[:100]}...')

## Part 4: Full RAG Pipeline

In [ ]:
def answer_question(
    question: str,
    retriever: HybridRetriever,
    model: str = 'gpt-4o-mini',
    k: int = 4,
    temperature: float = 0,
) -> dict:
    """
    Full RAG pipeline:
    1. Retrieve relevant chunks
    2. Build context
    3. Generate answer grounded in context
    """
    # 1. Retrieve
    retrieved = retriever.retrieve(question, k=k)

    # 2. Build context with source attribution
    context_parts = []
    sources = []
    for i, (chunk, score) in enumerate(retrieved, 1):
        context_parts.append(f'[Source {i}: {chunk.doc_title}]\n{chunk.text}')
        sources.append({'title': chunk.doc_title, 'score': score})

    context = '\n\n---\n\n'.join(context_parts)

    # 3. Generate answer
    system_prompt = '''You are a precise question-answering assistant.
Answer based ONLY on the provided context. Do not use outside knowledge.
If the answer is not in the context, say "I don't know based on the available documents."
Always cite the source number (e.g., [Source 1]) for key facts.'''

    user_prompt = f'''Context:
{context}

Question: {question}'''

    response = client.chat.completions.create(
        model=model,
        messages=[
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': user_prompt},
        ],
        temperature=temperature,
        max_tokens=500,
    )

    return {
        'question': question,
        'answer': response.choices[0].message.content,
        'sources': sources,
        'context': context,
        'tokens': response.usage.total_tokens,
    }


# Test the RAG pipeline
test_questions = [
    'Who created Python and when was it released?',
    'What are the three types of machine learning?',
    'What is RAG and why is it useful?',
    'What is the capital of France?',  # Should say "I don't know"
]

for question in test_questions:
    result = answer_question(question, retriever)
    print(f'\n{'='*60}')
    print(f'Q: {question}')
    print(f'A: {result["answer"]}')
    print(f'Sources: {[s["title"] for s in result["sources"][:2]]}')

## Part 5: Evaluation with RAGAS

In [ ]:
# pip install ragas
try:
    from ragas import evaluate
    from ragas.metrics import faithfulness, answer_relevancy, context_precision
    from datasets import Dataset

    # Build evaluation dataset
    eval_data = []
    ground_truths = [
        'Python was created by Guido van Rossum and first released in 1991.',
        'The three types are: supervised learning, unsupervised learning, and reinforcement learning.',
        'RAG combines LLMs with external knowledge retrieval to reduce hallucinations and enable access to current information.',
    ]

    for question, ground_truth in zip(test_questions[:3], ground_truths):
        result = answer_question(question, retriever)
        eval_data.append({
            'question': question,
            'answer': result['answer'],
            'contexts': [c.text for c, _ in retriever.retrieve(question, k=3)],
            'ground_truth': ground_truth,
        })

    dataset = Dataset.from_list(eval_data)

    print('Running RAGAS evaluation...')
    results = evaluate(
        dataset,
        metrics=[faithfulness, answer_relevancy, context_precision],
    )

    print('\nRAGAS EVALUATION RESULTS')
    print('=' * 40)
    print(f"Faithfulness:      {results['faithfulness']:.3f}  (>0.85 = good)")
    print(f"Answer Relevancy:  {results['answer_relevancy']:.3f}  (>0.85 = good)")
    print(f"Context Precision: {results['context_precision']:.3f}  (>0.80 = good)")

except ImportError:
    print('ragas not installed. Run: pip install ragas')
    print('Manual evaluation questions:')
    print('1. Is the answer grounded in the retrieved context? (faithfulness)')
    print('2. Does the answer address the question? (answer_relevancy)')
    print('3. Are the retrieved chunks relevant? (context_precision)')

## Summary: RAG Components Map

```
Document → Chunker → Embedder → Vector Store
                                     ↓
Query → Embedder → Similarity Search →┐
Query → BM25 Tokenizer → BM25 Search →┤
                                     ↓
                              RRF Fusion
                                     ↓
                         Context Assembly
                                     ↓
                         LLM (with context)
                                     ↓
                              Grounded Answer
```

## Challenge

1. **Load real documents:** Use `pypdf` to load a PDF and build a RAG system over it
2. **Parent-document retrieval:** Chunk at 100 chars, embed, but return the parent 500-char chunk
3. **HyDE:** Generate a hypothetical answer first, embed that, then search
4. **Reranking:** After retrieving top-10 chunks, use a cross-encoder to rerank and keep top-3